In [1]:
import pandas as pd

import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))


from src.labeling import add_returns
from src.volatility import add_close_to_close_volatility
from src.features import (
    add_lagged_returns,
    add_moving_average_features,
    add_volatility_features
)

# Load raw
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Merge HMM
reg = pd.read_csv("../data/processed/hmm_regimes.csv", parse_dates=["Date"])
df = df.merge(reg[["Date", "hmm_regime"]], on="Date", how="left")

# Feature pipeline
df = add_returns(df)
df = add_close_to_close_volatility(df)
df = add_lagged_returns(df)
df = add_moving_average_features(df)
df = add_volatility_features(df, vol_col="vol_cc")

# Targets
df["target_ret"] = df["return"].shift(-1)
df["target_dir"] = (df["target_ret"] > 0).astype(int)

features = [
    "ret_lag_1","ret_lag_5","ret_lag_10",
    "ma_ratio_5","ma_ratio_10","ma_ratio_20",
    "vol_cc","vol_cc_lag_1"
]

data = df.dropna(subset=features + ["target_ret","target_dir","hmm_regime"])

In [2]:
split = int(len(data) * 0.8)
train = data.iloc[:split]
test = data.iloc[split:]

In [3]:
import numpy as np

def create_sequences(df, feature_cols, target_col, window):
    X, y = [], []
    values = df[feature_cols].values
    targets = df[target_col].values
    
    for i in range(window, len(df)):
        X.append(values[i-window:i])
        y.append(targets[i])
    
    return np.array(X), np.array(y)

window = 20

# Regression sequences
X_train_reg, y_train_reg = create_sequences(train, features, "target_ret", window)
X_test_reg, y_test_reg = create_sequences(test, features, "target_ret", window)

# Classification sequences
X_train_cls, y_train_cls = create_sequences(train, features, "target_dir", window)
X_test_cls, y_test_cls = create_sequences(test, features, "target_dir", window)

In [4]:
import torch

# Regression tensors
X_train_reg_t = torch.tensor(X_train_reg, dtype=torch.float32)
y_train_reg_t = torch.tensor(y_train_reg, dtype=torch.float32)
X_test_reg_t = torch.tensor(X_test_reg, dtype=torch.float32)
y_test_reg_t = torch.tensor(y_test_reg, dtype=torch.float32)

# Classification tensors
X_train_cls_t = torch.tensor(X_train_cls, dtype=torch.float32)
y_train_cls_t = torch.tensor(y_train_cls, dtype=torch.float32)
X_test_cls_t = torch.tensor(X_test_cls, dtype=torch.float32)
y_test_cls_t = torch.tensor(y_test_cls, dtype=torch.float32)

# **LSTM Classification Model**

In [5]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

In [6]:
model_cls = LSTMClassifier(input_size=len(features))
optimizer_cls = torch.optim.Adam(model_cls.parameters(), lr=0.001)
criterion_cls = nn.BCEWithLogitsLoss()

epochs = 25

for epoch in range(epochs):
    model_cls.train()
    optimizer_cls.zero_grad()
    
    logits = model_cls(X_train_cls_t)
    loss = criterion_cls(logits, y_train_cls_t)
    
    loss.backward()
    optimizer_cls.step()
    
    if epoch % 5 == 0:
        print(f"[CLS] Epoch {epoch}, Loss: {loss.item():.6f}")

[CLS] Epoch 0, Loss: 0.695299
[CLS] Epoch 5, Loss: 0.692473
[CLS] Epoch 10, Loss: 0.691937
[CLS] Epoch 15, Loss: 0.692187
[CLS] Epoch 20, Loss: 0.691894


In [8]:
model_cls.eval()
with torch.no_grad():
    logits_test = model_cls(X_test_cls_t)
    probs_test = torch.sigmoid(logits_test)
    preds_test = (probs_test > 0.5).int()

accuracy_cls = (preds_test.squeeze() == y_test_cls_t.int()).float().mean().item()

print("LSTM Classification Accuracy:", accuracy_cls)

LSTM Classification Accuracy: 0.5443181991577148


# **LSTM Regression Model**

In [9]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

In [10]:
model_reg = LSTMRegressor(input_size=len(features))
optimizer_reg = torch.optim.Adam(model_reg.parameters(), lr=0.001)
criterion_reg = nn.MSELoss()

for epoch in range(epochs):
    model_reg.train()
    optimizer_reg.zero_grad()
    
    preds = model_reg(X_train_reg_t)
    loss = criterion_reg(preds, y_train_reg_t)
    
    loss.backward()
    optimizer_reg.step()
    
    if epoch % 5 == 0:
        print(f"[REG] Epoch {epoch}, Loss: {loss.item():.6f}")

[REG] Epoch 0, Loss: 0.000509
[REG] Epoch 5, Loss: 0.000212
[REG] Epoch 10, Loss: 0.000220
[REG] Epoch 15, Loss: 0.000228
[REG] Epoch 20, Loss: 0.000222


In [11]:
from sklearn.metrics import mean_squared_error

model_reg.eval()
with torch.no_grad():
    preds_reg_test = model_reg(X_test_reg_t).numpy()

mse_reg = mean_squared_error(y_test_reg, preds_reg_test)
corr_reg = np.corrcoef(y_test_reg, preds_reg_test)[0,1]

# Derived direction accuracy
direction_reg = (preds_reg_test > 0).astype(int)
direction_true = (y_test_reg > 0).astype(int)

accuracy_reg = (direction_reg == direction_true).mean()

print("LSTM Regression MSE:", mse_reg)
print("LSTM Regression Corr:", corr_reg)
print("LSTM Regression Direction Accuracy:", accuracy_reg)

LSTM Regression MSE: 7.117036162616163e-05
LSTM Regression Corr: -0.017484453135307273
LSTM Regression Direction Accuracy: 0.5443181818181818


# **Regime Conditioning**

In [13]:
test_regimes = test.iloc[window:]["hmm_regime"].values

import pandas as pd

results_cls = pd.DataFrame({
    "regime": test_regimes,
    "correct": (preds_test.squeeze().numpy() == y_test_cls)
})

results_reg = pd.DataFrame({
    "regime": test_regimes,
    "correct": (direction_reg == direction_true)
})

print("Classification Regime Accuracy:")
print(results_cls.groupby("regime")["correct"].mean())
print("\n")
print("Regression Regime Accuracy:")
print(results_reg.groupby("regime")["correct"].mean())

Classification Regime Accuracy:
regime
High      0.681818
Low       0.559809
Medium    0.514354
Name: correct, dtype: float64


Regression Regime Accuracy:
regime
High      0.681818
Low       0.557416
Medium    0.516746
Name: correct, dtype: float64
